# Knowledge Graph Memory

> **Extract relationship triples (subject-predicate-object) from every turn, build an in-memory graph with NetworkX, and query multi-hop paths to answer questions that require connecting facts scattered across the conversation.**

Think of a family tree or an org chart. These diagrams don't store facts about one person in isolation. They show how people connect: "Alice manages Bob," "Bob works on Project X," "Project X depends on the Q3 budget." You can trace paths through the connections to answer questions that no single fact could answer alone.

Previous memory techniques store information *about* individual entities (entity memory) or *about* the conversation itself (buffer, summary, vector store). None of them capture the **relationships between entities** in a structured, traversable way.

**Knowledge Graph Memory** takes a different approach. After each turn, the system extracts triples (`(Alice, manages, Bob)`, `(Project Alpha, depends_on, Q3 Budget)`) and adds them to a directed graph. When the user asks "Who are all the people working on projects that Alice manages?", the system traverses the graph. It follows edges across multiple hops to assemble an answer no single fact could provide.

**By the end of this notebook you'll understand:**
- How to build a triple store backed by NetworkX's directed graph.
- How to extract relationship triples from conversation using Claude's tool-use API.
- How subgraph retrieval injects only relevant relationship context into prompts.
- Multi-hop reasoning: answering questions that span multiple connected facts.
- How knowledge graph memory compares to simpler approaches on relationship-heavy recall tasks.

## Key Concepts

- **Triple (subject-predicate-object):** The atomic unit of a knowledge graph. Every relationship is expressed as a directed edge: `(subject) --[predicate]--> (object)`. For example: `(Alice, manages, Bob)` or `(Wavelength, uses, Python)`.
- **Knowledge graph:** A directed, labeled graph where nodes are entities and edges are typed relationships. Unlike a flat entity store, the graph captures *how* things relate to each other.
- **Triple extraction:** Using an LLM (via tool-use / function calling) to parse free-text conversation into structured triples. More flexible than rule-based approaches, but noisier.
- **Subgraph retrieval:** Given a query, you identify mentioned entities and extract their local neighborhood (1-2 hops away) from the full graph. Only the relevant subgraph goes into the prompt.
- **Multi-hop reasoning:** Answering questions that require traversing multiple edges. For example, "What projects depend on the budget that Alice controls?" requires following `Alice -> controls -> Budget -> depended_on_by -> Project`.
- **Graph persistence:** Serializing the graph (e.g., as a JSON edge list) for cross-session storage. Relationship knowledge survives across conversations.

## Architecture

<p align="center">
  <img src="../../images/diagrams/08_knowledge_graph_memory.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    subgraph Turn["Each Conversation Turn"]
        A["User message"] --> B["Triple Extractor\n(Claude tool-use)"]
        B --> C["Extracted triples\n(S, P, O)"]
        C --> D["Update\nKnowledge Graph"]
    end

    subgraph Graph["Knowledge Graph (NetworkX DiGraph)"]
        D --> E[("Nodes = entities\nEdges = predicates\n(Alice)--manages-->(Bob)")]
    end

    subgraph Response["Response Generation"]
        A --> F["Identify mentioned\nentities"]
        E --> F
        F --> G["Retrieve subgraph\n(1-2 hop neighborhood)"]
        G --> H["Build prompt:\nsystem + graph context\n+ recent messages"]
        H --> I["LLM\n(Claude)"]
        I --> J["Response"]
    end

    subgraph Persist["Persistence"]
        E --> K["Save edge list\n(JSON)"]
        K --> L["Load in\nnew session"]
        L --> E
    end

    style E fill:#4f46e5,color:#fff
    style I fill:#059669,color:#fff
    style B fill:#d97706,color:#fff
```

</details>

In [ ]:
# Install required packages (run once)
%pip install -q anthropic python-dotenv matplotlib numpy networkx

Load environment variables and initialize clients. We use Anthropic's Claude for chat and triple extraction, and NetworkX for the in-memory graph.

In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()  # reads API keys from .env

import anthropic
import networkx as nx

assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your .env file"

print("\u2713 API key loaded")
print(f"\u2713 anthropic version: {anthropic.__version__}")
print(f"\u2713 networkx version: {nx.__version__}")

## Core Implementation

Our Knowledge Graph Memory system has three components:

1. **`TripleStore`:** A NetworkX `DiGraph`-backed store where nodes are entities and edges carry a `predicate` label. Supports adding triples, querying neighborhoods, finding paths, and serialization.
2. **`TripleExtractor`:** Uses Claude's tool-use API to parse free-text into `(subject, predicate, object)` triples. The LLM acts as a relationship parser via function calling.
3. **`KnowledgeGraphMemory`:** The main class that orchestrates extraction, graph updates, subgraph retrieval, context injection, and chat.

In [ ]:
class TripleStore:
    """Triple store backed by a NetworkX directed graph."""

    def __init__(self):
        self.graph = nx.DiGraph()

    def add_triple(self, subject: str, predicate: str, obj: str) -> None:
        """Add a (subject, predicate, object) triple to the graph."""
        s, o = subject.strip(), obj.strip()
        p = predicate.strip().lower().replace(" ", "_")

        self.graph.add_node(s)
        self.graph.add_node(o)

        # If an identical edge exists, skip; otherwise update
        existing = self.graph.get_edge_data(s, o)
        if existing and existing.get("predicate") == p:
            return
        self.graph.add_edge(s, o, predicate=p)

    def get_triples(self) -> list[tuple[str, str, str]]:
        """Return all triples as (subject, predicate, object) tuples."""
        return [
            (u, data["predicate"], v)
            for u, v, data in self.graph.edges(data=True)
        ]

    def get_entity_triples(self, entity: str) -> list[tuple[str, str, str]]:
        """Get all triples involving a specific entity (as subject or object)."""
        triples = []
        e = entity.strip()
        for _, v, data in self.graph.out_edges(e, data=True):
            triples.append((e, data["predicate"], v))
        for u, _, data in self.graph.in_edges(e, data=True):
            triples.append((u, data["predicate"], e))
        return triples

    def get_neighborhood(self, entities: list[str], depth: int = 2) -> list[tuple[str, str, str]]:
        """Retrieve the subgraph within `depth` hops of the given entities."""
        relevant_nodes = set()
        for entity in entities:
            e = entity.strip()
            if e not in self.graph:
                continue
            undirected = self.graph.to_undirected()
            reachable = nx.single_source_shortest_path_length(undirected, e, cutoff=depth)
            relevant_nodes.update(reachable.keys())

        triples = []
        for u, v, data in self.graph.edges(data=True):
            if u in relevant_nodes or v in relevant_nodes:
                triples.append((u, data["predicate"], v))
        return triples



Next we add graph traversal methods. `find_path` uses NetworkX's shortest-path algorithm to trace a route between two entities through the graph. `find_mentioned` scans text for known entity names. `format_triples` turns a list of triples into readable text for prompt injection.

In [ ]:
    def find_path(self, source: str, target: str) -> list[tuple[str, str, str]] | None:
        """Find a path between two entities, returned as a list of triples."""
        s, t = source.strip(), target.strip()
        undirected = self.graph.to_undirected()
        try:
            path_nodes = nx.shortest_path(undirected, s, t)
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            return None

        triples = []
        for i in range(len(path_nodes) - 1):
            u, v = path_nodes[i], path_nodes[i + 1]
            if self.graph.has_edge(u, v):
                p = self.graph[u][v]["predicate"]
                triples.append((u, p, v))
            elif self.graph.has_edge(v, u):
                p = self.graph[v][u]["predicate"]
                triples.append((v, p, u))
        return triples

    def find_mentioned(self, text: str) -> list[str]:
        """Find all graph entities mentioned in a text string."""
        text_lower = text.lower()
        return [node for node in self.graph.nodes if node.lower() in text_lower]

    def format_triples(self, triples: list[tuple[str, str, str]]) -> str:
        """Format triples as human-readable text for prompt injection."""
        if not triples:
            return "No known relationships."
        seen = set()
        lines = []
        for s, p, o in triples:
            key = (s, p, o)
            if key not in seen:
                seen.add(key)
                lines.append(f"  {s} --[{p}]--> {o}")
        return "\n".join(lines)



We add persistence methods. The `save` method writes the graph as a JSON edge list. The `load` class method rebuilds the graph from that file. This lets the agent's relationship knowledge survive between sessions.

In [ ]:
    def save(self, path: str) -> None:
        """Persist graph as a JSON edge list."""
        data = {
            "nodes": list(self.graph.nodes),
            "edges": [
                {"subject": u, "predicate": d["predicate"], "object": v}
                for u, v, d in self.graph.edges(data=True)
            ],
        }
        with open(path, "w") as f:
            json.dump(data, f, indent=2)

    @classmethod
    def load(cls, path: str) -> "TripleStore":
        """Load graph from a JSON edge list."""
        store = cls()
        with open(path) as f:
            data = json.load(f)
        for edge in data["edges"]:
            store.add_triple(edge["subject"], edge["predicate"], edge["object"])
        return store

    def __len__(self) -> int:
        return self.graph.number_of_edges()

    def __repr__(self) -> str:
        return f"TripleStore({self.graph.number_of_nodes()} nodes, {self.graph.number_of_edges()} edges)"


print("\u2713 TripleStore class defined (NetworkX-backed)")

We define the tool schema for triple extraction. This JSON schema tells Claude how to call our `store_triples` function. It specifies three required fields per triple: subject, predicate (the relationship type), and object.

In [ ]:
# Define the tool schema for triple extraction via Claude function calling
EXTRACT_TRIPLES_TOOL = {
    "name": "store_triples",
    "description": (
        "Extract relationship triples (subject-predicate-object) from the conversation. "
        "Each triple represents a relationship between two entities. "
        "Call this tool with ALL relationships mentioned or implied."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "triples": {
                "type": "array",
                "description": "List of relationship triples extracted from the message.",
                "items": {
                    "type": "object",
                    "properties": {
                        "subject": {
                            "type": "string",
                            "description": "The source entity (e.g., 'Alice', 'Project Alpha').",
                        },
                        "predicate": {
                            "type": "string",
                            "description": "The relationship type (e.g., 'manages', 'works_at', 'depends_on').",
                        },
                        "object": {
                            "type": "string",
                            "description": "The target entity (e.g., 'Bob', 'Acme Corp').",
                        },
                    },
                    "required": ["subject", "predicate", "object"],
                },
            },
        },
        "required": ["triples"],
    },
}




The `TripleExtractor` sends each conversation exchange to Claude with the tool schema above. Claude parses the text and returns structured triples. We use snake_case predicates (like `works_at`, `manages`) for consistency.

In [ ]:
class TripleExtractor:
    """Uses Claude tool-use to extract relationship triples from messages."""

    def __init__(self, model: str = "claude-sonnet-4-20250514"):
        self.client = anthropic.Anthropic()
        self.model = model

    def extract(self, user_msg: str, assistant_msg: str) -> list[dict]:
        """Extract triples from a user-assistant exchange.

        Returns a list of dicts with keys: subject, predicate, object.
        """
        response = self.client.messages.create(
            model=self.model,
            max_tokens=1024,
            system=(
                "You are a relationship extraction system. Analyze the conversation "
                "and extract ALL relationships as (subject, predicate, object) triples. "
                "Use concise, consistent entity names (e.g., 'Alice' not 'my friend Alice'). "
                "Use snake_case predicates (e.g., 'works_at', 'manages', 'located_in'). "
                "Always call the store_triples tool. If no relationships are found, "
                "call it with an empty list."
            ),
            messages=[
                {
                    "role": "user",
                    "content": (
                        f"Extract relationship triples from this exchange:\n\n"
                        f"User: {user_msg}\n"
                        f"Assistant: {assistant_msg}"
                    ),
                }
            ],
            tools=[EXTRACT_TRIPLES_TOOL],
            tool_choice={"type": "tool", "name": "store_triples"},
        )

        for block in response.content:
            if block.type == "tool_use" and block.name == "store_triples":
                return block.input.get("triples", [])
        return []


print("\u2713 TripleExtractor class defined (uses Claude tool-use)")

The `KnowledgeGraphMemory` class orchestrates everything. When building the prompt, it finds entities mentioned in the user's message, retrieves their graph neighborhood (all triples within 2 hops), and injects that subgraph as context.

In [ ]:
class KnowledgeGraphMemory:
    """Chat agent with knowledge graph memory: extract triples, build a graph,
    and inject relevant subgraph context into each prompt."""

    def __init__(
        self,
        chat_model: str = "claude-sonnet-4-20250514",
        extraction_model: str = "claude-sonnet-4-20250514",
        system_prompt: str | None = None,
        max_tokens: int = 1024,
        recent_buffer_size: int = 10,
        neighborhood_depth: int = 2,
        triple_store: TripleStore | None = None,
    ):
        self.client = anthropic.Anthropic()
        self.chat_model = chat_model
        self.base_system_prompt = system_prompt or "You are a helpful assistant."
        self.max_tokens = max_tokens
        self.recent_buffer_size = recent_buffer_size
        self.neighborhood_depth = neighborhood_depth

        # Components
        self.store = triple_store or TripleStore()
        self.extractor = TripleExtractor(model=extraction_model)

        # State
        self.messages: list[dict] = []
        self.turn_count = 0

    def _build_system_prompt(self, user_input: str) -> str:
        """Inject relevant graph context into the system prompt."""
        parts = [self.base_system_prompt]

        # Find entities mentioned in the current message
        mentioned = self.store.find_mentioned(user_input)
        if mentioned:
            neighborhood = self.store.get_neighborhood(
                mentioned, depth=self.neighborhood_depth
            )
            if neighborhood:
                graph_context = self.store.format_triples(neighborhood)
                parts.append(
                    f"You have the following relationship knowledge relevant "
                    f"to the user's message:\n\n{graph_context}\n\n"
                    f"Use these relationships to give informed, accurate responses. "
                    f"You can follow chains of relationships to answer multi-hop questions."
                )

        # Brief overview of full graph
        all_triples = self.store.get_triples()
        if all_triples:
            entities = sorted(set(
                e for s, p, o in all_triples for e in (s, o)
            ))
            parts.append(f"All known entities: {', '.join(entities)}")

        return "\n\n".join(parts)



The `chat` method is the main loop. It builds the system prompt with graph context, calls Claude, then extracts new triples from the exchange and adds them to the graph.

In [ ]:
    def chat(self, user_input: str) -> str:
        """Send a message, extract triples, update graph, and respond with context."""
        self.turn_count += 1

        system = self._build_system_prompt(user_input)

        self.messages.append({"role": "user", "content": user_input})
        recent = self.messages[-self.recent_buffer_size:]

        response = self.client.messages.create(
            model=self.chat_model,
            max_tokens=self.max_tokens,
            system=system,
            messages=recent,
        )
        assistant_text = response.content[0].text

        self.messages.append({"role": "assistant", "content": assistant_text})

        # Extract triples from this exchange
        try:
            extracted = self.extractor.extract(user_input, assistant_text)
            for triple in extracted:
                self.store.add_triple(
                    triple["subject"], triple["predicate"], triple["object"]
                )
        except Exception as e:
            print(f"  [extraction warning: {e}]")

        return assistant_text



We add utility methods. `query_graph` returns all relationships for a single entity. `find_connection` traces the shortest path between two entities. `save_memory` and `from_saved` handle cross-session persistence.

In [ ]:
    def query_graph(self, entity: str) -> str:
        """Query all known relationships for an entity."""
        triples = self.store.get_entity_triples(entity)
        if not triples:
            return f"No known relationships for '{entity}'."
        return self.store.format_triples(triples)

    def find_connection(self, entity1: str, entity2: str) -> str:
        """Find the shortest path between two entities."""
        path = self.store.find_path(entity1, entity2)
        if path is None:
            return f"No connection found between '{entity1}' and '{entity2}'."
        return self.store.format_triples(path)

    def save_memory(self, path: str) -> None:
        """Persist knowledge graph to disk."""
        self.store.save(path)

    @classmethod
    def from_saved(cls, path: str, **kwargs) -> "KnowledgeGraphMemory":
        """Create instance pre-loaded with a saved knowledge graph."""
        store = TripleStore.load(path)
        return cls(triple_store=store, **kwargs)

    def __repr__(self) -> str:
        return (
            f"KnowledgeGraphMemory(turns={self.turn_count}, "
            f"nodes={self.store.graph.number_of_nodes()}, "
            f"edges={self.store.graph.number_of_edges()})"
        )


print("\u2713 KnowledgeGraphMemory class defined")

## Usage Example: Building a Relationship Graph

Let's have a multi-turn conversation that establishes a web of relationships between people, projects, and places. After each turn, the system extracts triples and grows the knowledge graph. Later turns benefit from graph-based context injection.

In [ ]:
mem = KnowledgeGraphMemory(
    system_prompt="You are a friendly personal assistant. Reply concisely in 1-2 sentences.",
)

messages = [
    "Hi! I'm Priya. I work as a data scientist at Spotify.",
    "My manager is Carlos -- he leads the recommendations team.",
    "Carlos also manages Aisha, who's a machine learning engineer.",
    "I'm working on Project Wavelength with Aisha. It's a music recommendation engine.",
    "Wavelength depends on a dataset called MusicGraph that the data engineering team maintains.",
    "My partner Alex is a chef -- he runs a restaurant called Basil & Lime in Portland.",
    "How are Aisha and Carlos related?",
    "What do you know about Project Wavelength and its dependencies?",
]

for msg in messages:
    print(f"\U0001f464 User:  {msg}")
    reply = mem.chat(msg)
    print(f"\U0001f916 Agent: {reply}")
    print()

print(f"\n\U0001f4ca Graph: {mem.store.graph.number_of_nodes()} nodes, {mem.store.graph.number_of_edges()} edges")

Let's inspect the knowledge graph. We print every triple (relationship), list all known entities, query a specific entity's connections, and trace a path between two entities.

In [ ]:
# Inspect the knowledge graph
print("=== All Triples in the Knowledge Graph ===\n")
for s, p, o in mem.store.get_triples():
    print(f"  ({s}) --[{p}]--> ({o})")

print(f"\n=== Entities: {sorted(mem.store.graph.nodes)} ===")

# Query specific entity relationships
print("\n=== Relationships for 'Carlos' ===")
print(mem.query_graph("Carlos"))

# Find connections between entities
print("\n=== Path: Priya \u2192 MusicGraph ===")
print(mem.find_connection("Priya", "MusicGraph"))

## Visualizing the Knowledge Graph

One advantage of graph-based memory: you can directly visualize the relationships the agent has learned.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(12, 7))

G = mem.store.graph
pos = nx.spring_layout(G, seed=42, k=2.5)

# Draw nodes
nx.draw_networkx_nodes(G, pos, ax=ax, node_color="#4f46e5", node_size=1800, alpha=0.9)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=9, font_color="white", font_weight="bold")

# Draw edges with labels
edge_labels = {(u, v): d["predicate"] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#888", arrows=True,
                       arrowsize=20, connectionstyle="arc3,rad=0.1", width=1.5)
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax,
                             font_size=7, font_color="#d97706")

ax.set_title("Knowledge Graph: Extracted Relationships", fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig("knowledge_graph.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\u2713 Graph visualization saved (knowledge_graph.png)")

## Multi-Hop Reasoning

This is the key advantage of graph memory over entity memory: answering questions that require **traversing multiple relationships**. The agent follows chains of edges to connect facts that were mentioned in entirely different turns.

In [ ]:
# Multi-hop questions that require connecting multiple triples
multi_hop_questions = [
    "What dataset does the project I'm working on depend on?",
    "Who else does my manager manage besides me?",
    "What's the connection between Alex and Spotify?",
]

print("=== Multi-Hop Reasoning ===\n")
for question in multi_hop_questions:
    print(f"\U0001f464 User:  {question}")
    reply = mem.chat(question)
    print(f"\U0001f916 Agent: {reply}")
    print()

## Cross-Session Persistence

The knowledge graph serializes to a JSON edge list. A new session loads the graph and immediately has access to all previously learned relationships. The user never needs to re-explain their world.

In [ ]:
# -- Save knowledge graph from Session 1 --
KG_FILE = "knowledge_graph.json"
mem.save_memory(KG_FILE)
print(f"\u2713 Saved graph ({len(mem.store)} triples) to {KG_FILE}")

# -- Simulate a new session (Session 2) --
print("\n--- NEW SESSION ---\n")

mem2 = KnowledgeGraphMemory.from_saved(
    KG_FILE,
    system_prompt="You are a friendly personal assistant. Reply concisely in 1-2 sentences.",
)
print(f"Loaded graph: {mem2.store.graph.number_of_nodes()} nodes, {mem2.store.graph.number_of_edges()} edges")
print(f"Known entities: {sorted(mem2.store.graph.nodes)}\n")

session2_questions = [
    "Who manages the recommendations team?",
    "What project am I working on, and who's on it with me?",
    "Tell me about Alex.",
]

for msg in session2_questions:
    print(f"\U0001f464 User:  {msg}")
    reply = mem2.chat(msg)
    print(f"\U0001f916 Agent: {reply}")
    print()

print("\u2713 Cross-session persistence works -- relationships preserved!")

## Experiment: Knowledge Graph Memory vs. Sliding Window

We'll run a controlled test focused on **relationship-heavy recall**:

1. Plant **10 relationship facts** across a **30-turn conversation** (mixed with filler).
2. Ask **10 recall questions**, several requiring multi-hop reasoning.
3. Compare **Knowledge Graph Memory** (extracts & stores triples) vs. **Sliding Window** (keeps last 10 messages).

The hypothesis: knowledge graph memory will excel at relationship and multi-hop questions. The sliding window will struggle with facts that have left the window.

In [ ]:
# Relationship facts planted at specific turns
PLANTED_FACTS = {
    1:  ("My name is Jordan and I'm a software engineer at Nexus Corp.",         "jordan",       "engineer"),
    3:  ("My team lead is Sana -- she also mentors our intern, Leo.",             "sana",         "team_lead"),
    5:  ("Leo is working on Project Beacon, which is a real-time analytics platform.", "beacon",  "project"),
    8:  ("Project Beacon depends on a service called DataStream that Raj built.",  "datastream",   "dependency"),
    11: ("Raj works in the infrastructure team, which Sana used to lead.",         "raj",          "infra"),
    15: ("My partner Morgan is a teacher at Oakwood Elementary.",                  "morgan",       "partner"),
    18: ("Morgan coaches the school's debate team with a colleague named Dana.",   "dana",         "colleague"),
    22: ("Dana is also Leo's older sister -- small world!",                        "dana",         "family"),
    26: ("I'm thinking of taking a course on graph databases taught by Prof. Chen.", "chen",       "professor"),
    29: ("Prof. Chen wrote the textbook we use at Nexus Corp for onboarding.",      "chen",        "textbook"),
}

FILLER_MESSAGES = [
    "What's the difference between HTTP and HTTPS?",
    "Can you explain quantum entanglement simply?",
    "What's a good recipe for banana bread?",
    "How does a heat pump work?",
    "Tell me about the history of jazz.",
    "What's the tallest mountain in South America?",
    "How do noise-canceling headphones work?",
    "What causes auroras?",
    "Explain the Pythagorean theorem.",
    "What's the oldest known civilization?",
    "How does sourdough starter work?",
    "What is the Doppler effect?",
    "Tell me about coral reefs.",
    "How does memory work in the brain?",
    "What's the speed of light?",
    "How do electric cars regenerate braking energy?",
    "What is dark energy?",
    "Tell me about the Silk Road trade routes.",
    "How do birds navigate during migration?",
    "What is the Krebs cycle?",
]



We build the 30-turn conversation by interleaving planted relationship facts with filler. Then we define recall questions. Several questions require multi-hop reasoning: connecting facts from different turns through the graph.

In [ ]:
# Build 30-turn conversation
conversation_30 = []
filler_idx = 0
for turn in range(1, 31):
    if turn in PLANTED_FACTS:
        msg = PLANTED_FACTS[turn][0]
    else:
        msg = FILLER_MESSAGES[filler_idx % len(FILLER_MESSAGES)]
        filler_idx += 1
    conversation_30.append((turn, msg))

# Some questions require multi-hop graph traversal
RECALL_QUESTIONS = [
    ("Where do I work?",                                          "nexus",       1),
    ("Who is my team lead?",                                      "sana",        3),
    ("What project is Leo working on?",                           "beacon",      5),
    ("What does Project Beacon depend on?",                       "datastream",  8),
    ("What team does Raj work on?",                               "infrastructure", 11),
    ("Who is Morgan?",                                            "teacher",     15),
    ("Who does Morgan coach the debate team with?",               "dana",        18),
    ("How are Dana and Leo related?",                             "sister",      22),
    ("Who teaches the graph databases course?",                   "chen",        26),
    ("Who wrote the textbook used for onboarding at Nexus Corp?", "chen",        29),
]

print(f"Conversation: {len(conversation_30)} turns")
print(f"Facts planted at turns: {sorted(PLANTED_FACTS.keys())}")
print(f"Recall questions: {len(RECALL_QUESTIONS)}")

We run all 30 turns through the knowledge graph memory. After each turn, the extractor identifies relationship triples and adds them to the graph. Watch the node and edge counts grow.

In [ ]:
# -- Run 30 turns through Knowledge Graph Memory --

kg_mem = KnowledgeGraphMemory(
    system_prompt="You are a helpful assistant. Reply concisely in 1-2 sentences.",
)

print("Running 30-turn conversation through Knowledge Graph Memory...")
for turn_num, msg in conversation_30:
    kg_mem.chat(msg)
    if turn_num % 10 == 0:
        print(f"  Turn {turn_num}/30 done ({kg_mem.store.graph.number_of_nodes()} nodes, {len(kg_mem.store)} edges)")

print(f"\n\u2713 Done. Graph: {kg_mem.store.graph.number_of_nodes()} nodes, {len(kg_mem.store)} edges")
print(f"\n=== Extracted Triples ===")
for s, p, o in kg_mem.store.get_triples():
    print(f"  ({s}) --[{p}]--> ({o})")

We define a sliding window baseline and run the same 30 turns through it. The window keeps only the last 10 messages, so older relationship facts will drop out of context.

In [ ]:
class SlidingWindowBaseline:
    """Sliding window: keeps only the last K messages."""

    def __init__(self, window_size: int = 10, model: str = "claude-sonnet-4-20250514",
                 system_prompt: str | None = None, max_tokens: int = 1024):
        self.window_size = window_size
        self.client = anthropic.Anthropic()
        self.model = model
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self.messages: list[dict] = []

    def chat(self, user_input: str) -> str:
        self.messages.append({"role": "user", "content": user_input})
        window = self.messages[-self.window_size:]

        kwargs = dict(model=self.model, max_tokens=self.max_tokens, messages=window)
        if self.system_prompt:
            kwargs["system"] = self.system_prompt

        response = self.client.messages.create(**kwargs)
        assistant_text = response.content[0].text
        self.messages.append({"role": "assistant", "content": assistant_text})
        return assistant_text


# -- Run 30 turns through Sliding Window --

sw_mem = SlidingWindowBaseline(
    window_size=10,
    system_prompt="You are a helpful assistant. Reply concisely in 1-2 sentences.",
)

print("Running 30-turn conversation through Sliding Window Memory...")
for turn_num, msg in conversation_30:
    sw_mem.chat(msg)
    if turn_num % 10 == 0:
        print(f"  Turn {turn_num}/30 done")

print(f"\n\u2713 Done. Window keeps last {sw_mem.window_size} messages.")

We run the recall test on both systems. For each question, we check whether the answer contains the expected keyword. Knowledge graph memory should excel at multi-hop questions where the sliding window has long forgotten the relevant facts.

In [ ]:
# -- Recall test: Knowledge Graph Memory --
print("=== Knowledge Graph Memory - Recall Test ===\n")
kg_results = []

for question, keyword, planted_at in RECALL_QUESTIONS:
    answer = kg_mem.chat(question)
    recalled = keyword.lower() in answer.lower()
    kg_results.append({
        "question": question, "keyword": keyword,
        "planted_at": planted_at, "recalled": recalled,
        "answer": answer,
    })
    status = "\u2713" if recalled else "\u2717"
    print(f"  {status} (turn {planted_at:2d}) {question}")
    print(f"    Answer: {answer[:120]}")
    print()

# -- Recall test: Sliding Window --
print("=== Sliding Window - Recall Test ===\n")
sw_results = []

for question, keyword, planted_at in RECALL_QUESTIONS:
    answer = sw_mem.chat(question)
    recalled = keyword.lower() in answer.lower()
    sw_results.append({
        "question": question, "keyword": keyword,
        "planted_at": planted_at, "recalled": recalled,
        "answer": answer,
    })
    status = "\u2713" if recalled else "\u2717"
    print(f"  {status} (turn {planted_at:2d}) {question}")
    print(f"    Answer: {answer[:120]}")
    print()

kg_score = sum(1 for r in kg_results if r["recalled"])
sw_score = sum(1 for r in sw_results if r["recalled"])
print(f"Knowledge Graph Memory score:  {kg_score}/{len(RECALL_QUESTIONS)}")
print(f"Sliding Window score:          {sw_score}/{len(RECALL_QUESTIONS)}")

Let's visualize the comparison. Panel 1 shows total recall scores. Panel 2 breaks down recall by fact age.

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- Panel 1: Total recall scores --
kg_score = sum(1 for r in kg_results if r["recalled"])
sw_score = sum(1 for r in sw_results if r["recalled"])
labels = ["Knowledge Graph\nMemory", "Sliding Window\n(K=10)"]
scores = [kg_score, sw_score]
colors = ["#4f46e5", "#ef4444"]

bars = axes[0].bar(labels, scores, color=colors, width=0.5, alpha=0.85)
axes[0].set_ylabel("Facts Recalled")
axes[0].set_ylim(0, len(RECALL_QUESTIONS) + 1)
axes[0].set_title(f"Total Facts Recalled (out of {len(RECALL_QUESTIONS)})")
axes[0].axhline(y=len(RECALL_QUESTIONS), color="gray", linestyle="--", alpha=0.3)
for bar, score in zip(bars, scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(score), ha="center", fontweight="bold", fontsize=14)

# -- Panel 2: Recall by fact age --
fact_turns = [r["planted_at"] for r in kg_results]
kg_recalled = [1 if r["recalled"] else 0 for r in kg_results]
sw_recalled = [1 if r["recalled"] else 0 for r in sw_results]

x = np.arange(len(fact_turns))
width = 0.35
axes[1].bar(x - width/2, kg_recalled, width, label="KG Memory", color="#4f46e5", alpha=0.85)
axes[1].bar(x + width/2, sw_recalled, width, label="Sliding Window", color="#ef4444", alpha=0.85)
axes[1].set_xlabel("Fact Planted at Turn #")
axes[1].set_ylabel("Recalled? (1=Yes, 0=No)")
axes[1].set_title("Recall by Fact Age")
axes[1].set_xticks(x)
axes[1].set_xticklabels([str(t) for t in fact_turns], fontsize=9)
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(["No", "Yes"])
axes[1].legend()

plt.tight_layout()
plt.savefig("kg_vs_window.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nKnowledge Graph Memory:  {kg_score}/{len(RECALL_QUESTIONS)} recalled")
print(f"Sliding Window:         {sw_score}/{len(RECALL_QUESTIONS)} recalled")

## Discussion & Tradeoffs

### Strengths
- **Relationship-first.** Unlike entity memory (which stores facts *about* entities), knowledge graph memory captures facts *between* entities. This enables multi-hop reasoning that other approaches cannot support.
- **Traversable structure.** You can query the graph programmatically: shortest paths, connected components, neighborhood retrieval. No embedding similarity needed.
- **Visual interpretability.** The graph is directly visualizable. This makes it easy to inspect and debug what the agent "knows."
- **Composable with other techniques.** Use a knowledge graph alongside a sliding window (for recent context) and summary memory (for high-level recap).

### Weaknesses
- **Extraction noise.** LLM-based triple extraction is imperfect. Predicates may be inconsistent (`manages` vs. `is_manager_of`). Entity names may vary (`Professor Chen` vs. `Prof. Chen`). Spurious triples can appear from filler.
- **Double API cost.** Like entity memory, each turn requires an extra LLM call for extraction. This roughly doubles cost.
- **Graph maintenance.** Deduplication, normalization, conflict resolution, and pruning all require ongoing effort. Production systems need entity resolution and predicate canonicalization (standardizing relationship labels).
- **Serialization overhead.** Injecting graph context into prompts is less token-efficient than summary memory. Large graphs need aggressive subgraph filtering.
- **Scaling.** NetworkX is in-memory. For production graphs with millions of triples, use Neo4j or a similar graph database.

### Knowledge Graph Memory vs. Entity Memory

| Aspect | Entity Memory | Knowledge Graph Memory |
|--------|--------------|----------------------|
| Stores | Facts *about* entities | Relationships *between* entities |
| Structure | Key-value (flat) | Graph (nodes + edges) |
| Query | Lookup by name | Traversal, paths, neighborhoods |
| Multi-hop | Not supported | Core strength |
| Visualization | Entity list | Graph diagram |
| Complexity | Simpler | More complex |
| Best for | Personal facts, preferences | Org structures, dependencies |

### When to Use Knowledge Graph Memory

| Scenario | Recommendation |
|----------|---------------|
| Tracking complex relationships (org charts, project dependencies) | Excellent. The core use case. |
| Multi-hop questions spanning multiple facts | Excellent. |
| Personal assistant with individual facts | Overkill. Entity memory is sufficient. |
| Need to scale to millions of relationships | Use Neo4j instead of NetworkX. |
| Cost-sensitive deployments | Careful: extraction doubles per-turn cost. |
| Combining with other memory types | Great complement to entity memory + sliding window. |

## Further Reading

- [Anthropic Tool Use (Function Calling)](https://docs.anthropic.com/en/docs/build-with-claude/tool-use?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): The API used for triple extraction in this notebook
- [NetworkX: Python Graph Library](https://networkx.org/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): The in-memory graph backend used here
- [Neo4j Graph Database](https://neo4j.com/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Production-grade graph database for scaling beyond in-memory
- [Microsoft GraphRAG](https://github.com/microsoft/graphrag): Graph-based retrieval-augmented generation at scale
- [LangChain ConversationKGMemory](https://python.langchain.com/docs/modules/memory/types/kg?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Framework-level knowledge graph memory
- [Ji et al., "A Survey on Knowledge Graphs" (2022)](https://ieeexplore.ieee.org/document/9416312?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Survey on KG representation, acquisition, and applications
- [Anthropic: Building Conversational AI](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Multi-turn conversation patterns

---

*← Previous: [07 - Entity Memory](../07_entity_memory/) · Next: [09 - Associative Memory](../09_associative_memory/) →*

In [ ]:
# Clean up temp files
import os
for f in ["knowledge_graph.json", "knowledge_graph.png", "kg_vs_window.png"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: Confidence-scored triples
Add a `confidence` float (0.0 to 1.0) to each triple stored in `TripleStore`. Modify `get_neighborhood()` to accept a `min_confidence` parameter and filter out low-confidence edges. Test with a conversation where the user corrects earlier statements.

### Challenge 2: Graph structure metrics
After the 30-turn experiment, compute and print: total nodes, total edges, graph density (`nx.density()`), number of connected components, and average shortest path length within the largest component. Discuss what these numbers reveal about the conversation.

### Challenge 3: Temporal edge expiry
Add a `created_at` timestamp to each triple. Implement a `prune_stale()` method that removes triples older than a configurable threshold. Run 30 turns, prune edges older than the most recent 10 turns, and compare retrieval quality. This extends the idea from 18 Temporal Memory to graph structures.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--08-knowledge-graph-memory--knowledge-graph-memory)
